# Task 3 — GRPO training + evaluation (Colab)

Trained SFT adapter is already in this repo at `ckpts/sft/`.

**Run cells top-to-bottom.** GRPO cell is gated on the eval — do NOT run it until you see the SFT eval numbers and confirm SFT actually improved over baseline.


## 1. Setup

In [ ]:
!pip install -q 'transformers>=4.45' 'trl>=0.11' 'peft>=0.13' 'accelerate>=0.34' 'bitsandbytes>=0.43' 'datasets>=2.20' 'pydantic>=2.7'

# Clone the repo. Colab will prompt for GitHub auth on private repos.
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project
%cd /content/ml_project


In [ ]:
import sys, pathlib, torch
sys.path.insert(0, str(pathlib.Path.cwd()))
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Eval π_0 vs π_1 — DECIDES whether GRPO is worth running

~15-20 min on T4. Prints per-layout progress.

**Decision rule:**
- If `policy_1.mean_green_science > 0` and `policy_1.mean_composite > policy_0.mean_composite`: SFT worked → run GRPO.
- Otherwise: STOP. Debug SFT (more seeds, retrain).


In [ ]:
!python -m training.evaluate \
  --checkpoints policy_0=BASE policy_1=./ckpts/sft \
  --samples-per-layout 4 --n-val 20 \
  --out results/eval_sft_vs_base.json


In [ ]:
import json, pandas as pd
d = json.load(open('results/eval_sft_vs_base.json'))
rows = [{'ckpt': c['name'], 'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1)} for c in d]
pd.DataFrame(rows).set_index('ckpt')


## 3. GRPO training — RUN ONLY IF SFT IMPROVED

2-4 hours on T4. Saves checkpoints every 50 steps: `policy_2, policy_3, policy_final`.


In [ ]:
!python -m training.train_grpo \
  --init-adapter ./ckpts/sft \
  --output-dir ./ckpts/grpo \
  --max-steps 200 --save-steps 50


## 4. Final eval — all 5 checkpoints


In [ ]:
from pathlib import Path
from training.evaluate import evaluate_all
from dataclasses import asdict
import json

ckpt_dir = Path('./ckpts/grpo')
ckpt_paths = sorted((p for p in ckpt_dir.glob('checkpoint-*') if p.is_dir()),
                    key=lambda p: int(p.name.split('-')[-1]))
checkpoints = [('policy_0', None), ('policy_1', './ckpts/sft')]
for i, p in enumerate(ckpt_paths):
    name = 'policy_final' if i == len(ckpt_paths) - 1 else f'policy_{i+2}'
    checkpoints.append((name, str(p)))
print('Evaluating:', checkpoints)

metrics = evaluate_all(checkpoints, samples_per_layout=4, n_val=20)
Path('results').mkdir(exist_ok=True)
with open('results/checkpoint_eval.json', 'w') as f:
    json.dump([asdict(m) for m in metrics], f, indent=2)


## 5. Download results

After the last cell, download `results/checkpoint_eval.json` from the Colab file browser (folder icon on left) and put it in `results/` locally to run `notebooks/final_results.ipynb`.
